In [3]:
import pandas as pd
import os

subfolder_path = 'dataset/Genre Classification Dataset'
print("Files in subfolder:", os.listdir(subfolder_path))

# Load the dataset using the corrected nested path
file_path = f"{subfolder_path}/train_data.txt"
df = pd.read_csv(file_path, sep=':::', engine='python', names=['ID', 'Title', 'Genre', 'Description'])

# Display the first few rows and the dataset information
display(df.head())
print("\nDataset Info:")
df.info()

Files in subfolder: ['description.txt', 'test_data.txt', 'test_data_solution.txt', 'train_data.txt']


,ID,Title,Genre,Description
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his do...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous r...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fie...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends mee...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-re...



Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 54214 entries, 0 to 54213
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ID           54214 non-null  int64
 1   Title        54214 non-null  str  
 2   Genre        54214 non-null  str  
 3   Description  54214 non-null  str  
dtypes: int64(1), str(3)
memory usage: 34.6 MB


In [4]:
import re
import string

# Function to clean the text data
def clean_text(text):
    text = text.lower() # Convert to lowercase
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text) # Remove punctuation
    text = re.sub(r'\d+', ' ', text) # Remove numbers
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

# Clean the 'Genre' column just to ensure there are no hidden spaces
df['Genre'] = df['Genre'].str.strip()

# Apply the cleaning function to the 'Description' column
print("Cleaning text data. This might take a few seconds...")
df['Cleaned_Description'] = df['Description'].apply(clean_text)

# Display the original vs cleaned descriptions
display(df[['Genre', 'Description', 'Cleaned_Description']].head())

Cleaning text data. This might take a few seconds...


,Genre,Description,Cleaned_Description
0,drama,Listening in to a conversation between his do...,listening in to a conversation between his doc...
1,thriller,A brother and sister with a past incestuous r...,a brother and sister with a past incestuous re...
2,adult,As the bus empties the students for their fie...,as the bus empties the students for their fiel...
3,drama,To help their unemployed father make ends mee...,to help their unemployed father make ends meet...
4,drama,The film's title refers not only to the un-re...,the film s title refers not only to the un rec...


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Extracting features using TF-IDF. This may take a moment...")

# Initialize TF-IDF Vectorizer (limiting to top 10,000 words to optimize performance)
tfidf = TfidfVectorizer(max_features=10000)
X = tfidf.fit_transform(df['Cleaned_Description'])

# Encode the target labels (Genres) into numeric values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Genre'])

# Split the dataset into training and validation sets (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Feature matrix (X) shape: {X.shape}")
print(f"Target vector (y) shape: {y.shape}")
print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")

Extracting features using TF-IDF. This may take a moment...
Feature matrix (X) shape: (54214, 10000)
Target vector (y) shape: (54214,)
Training data shape: (43371, 10000)
Validation data shape: (10843, 10000)


In [6]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

print("Training the Naive Bayes model...")
# Initialize the Naive Bayes classifier
model = MultinomialNB()

# Train the model using the training data
model.fit(X_train, y_train)

# Make predictions on the validation data
y_pred = model.predict(X_val)

# Calculate and display the overall accuracy
accuracy = accuracy_score(y_val, y_pred)
print(f"\nValidation Accuracy: {accuracy * 100:.2f}%\n")

# Display a detailed classification report
# We use label_encoder.classes_ to map the numeric predictions back to the actual movie genres
print("Classification Report:")
print(classification_report(y_val, y_pred, target_names=label_encoder.classes_, zero_division=0))

Training the Naive Bayes model...

Validation Accuracy: 50.49%

Classification Report:
              precision    recall  f1-score   support

      action       0.60      0.01      0.02       263
       adult       0.67      0.02      0.03       112
   adventure       0.71      0.04      0.07       139
   animation       0.00      0.00      0.00       104
   biography       0.00      0.00      0.00        61
      comedy       0.52      0.40      0.45      1443
       crime       0.00      0.00      0.00       107
 documentary       0.56      0.90      0.69      2659
       drama       0.44      0.84      0.57      2697
      family       0.00      0.00      0.00       150
     fantasy       0.00      0.00      0.00        74
   game-show       0.00      0.00      0.00        40
     history       0.00      0.00      0.00        45
      horror       0.80      0.21      0.33       431
       music       1.00      0.01      0.03       144
     musical       0.00      0.00      0.00     

In [7]:
print("Loading test data...")
# Load the test dataset (which does not contain the 'Genre' column)
test_file_path = f"{subfolder_path}/test_data.txt"
df_test = pd.read_csv(test_file_path, sep=':::', engine='python', names=['ID', 'Title', 'Description'])

print("Cleaning test descriptions...")
# Apply the exact same cleaning function used on the training data
df_test['Cleaned_Description'] = df_test['Description'].apply(clean_text)

print("Extracting features and making predictions...")
# Transform the text using the ALREADY FITTED TF-IDF vectorizer
X_test = tfidf.transform(df_test['Cleaned_Description'])

# Make predictions on the unseen test data
test_predictions = model.predict(X_test)

# Convert the numeric predictions back into readable genre text
df_test['Predicted_Genre'] = label_encoder.inverse_transform(test_predictions)

print("Saving predictions to CSV...")
# Save the results to a new CSV file in your dataset folder
output_path = 'dataset/predicted_genres.csv'
df_test[['ID', 'Title', 'Predicted_Genre']].to_csv(output_path, index=False)

print(f"Task 1 Complete! Predictions successfully saved to {output_path}")
display(df_test[['ID', 'Title', 'Predicted_Genre']].head())

Loading test data...
Cleaning test descriptions...
Extracting features and making predictions...
Saving predictions to CSV...
Task 1 Complete! Predictions successfully saved to dataset/predicted_genres.csv


,ID,Title,Predicted_Genre
0,1,Edgar's Lunch (1998),drama
1,2,La guerra de papá (1977),drama
2,3,Off the Beaten Track (2010),documentary
3,4,Meu Amigo Hindu (2015),drama
4,5,Er nu zhai (1955),drama
